In [ ]:
!pip install faster-whisper gradio groq gtts

In [ ]:
import os
from groq import Groq
from faster_whisper import WhisperModel
from gtts import gTTS
import gradio as gr

# -------------------------
# Load Models
# -------------------------

# Whisper (tiny = fast, good for Colab)
whisper_model = WhisperModel("tiny")

# Groq client
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY")
)

# -------------------------
# Core Function
# -------------------------

def voice_ai(audio):

    # 1. Speech → Text
    segments, _ = whisper_model.transcribe(audio)
    user_text = " ".join([seg.text for seg in segments])

    if user_text.strip() == "":
        return "No speech detected", None

    print("User said:", user_text)

    # 2. Text → LLM (Groq)
    response = client.chat.completions.create(
        messages=[
            {"role": "user", "content": user_text}
        ],
        model="llama-3.3-70b-versatile",
    )

    ai_text = response.choices[0].message.content
    print("AI:", ai_text)

    # 3. Text → Speech (gTTS)
    tts = gTTS(ai_text)
    tts.save("output.mp3")

    return ai_text, "output.mp3"

# -------------------------
# Gradio UI
# -------------------------

app = gr.Interface(
    fn=voice_ai,
    inputs=gr.Audio(type="filepath"),
    outputs=[
        gr.Textbox(label="AI Response"),
        gr.Audio(label="Voice Output")
    ],
    title="Voice to Voice AI (Whisper + Groq + gTTS)"
)

app.launch(debug=True)